# REVIEW

For increases interpretability, the signal_info.py. file can more clearly defines: How each individual signal is calculated. How the signals are combined into the final stock score (A, B, C, etc.). Is it a simple weighted average? A more complex rule-based system? This combination logic should be very clear and explicit. Add comments that explain the weightings and rationale.

The thresholds for assigning letter grades (A, B, C, etc.). What score constitutes an "A" versus a "B"?

Centralize Signal Logic: Ideally, all signal calculation and combination logic should be centralized within signal_info.py or a dedicated "signals" module. Avoid scattering signal-related code throughout the notebook.

2. Deeper Dive into portfolio.ipynb:
Document compute_trades() Thoroughly: The compute_trades() method within the Portfolio class (defined in portfolio.ipynb) is where the buy/sell decisions are made. 
Add extensive comments to explain how the stock scores are used to determine which stocks to long and short, how the quantities of each stock to buy or sell are calculated. How is the equal weighting achieved? How is the leverage managed?
How transaction costs are handled (if at all). Also add any logic for filtering or excluding certain stocks.
We should also explain execute_trades(): The execute_trades() method simulates the trading process. Document how the simulation works, how prices are obtained, and how the portfolio is updated.

3. Make Data Processing more clear: Add comments to explain each step in the data processing pipeline. What missing values are handled, and how? Are any transformations applied to the data (e.g., scaling, normalization)


# Intro

IMPORTANT: Edit input info in signal_info.py

In this project, we will be testing a broken down buy/hold/sell strategy. This strategy forms a new portfolio every month by buying "A" stock and selling "F" stocks.

    # strategy: long 'A' stocks (>90 score), short 'F' stocks (<= 70 score for now)
    # For each $1 NAV, we open $1 total of long positions AND $1 total of short positions
    # This would be the max leverage allowed given 50% margin requirements
    # Assume 100% of portfolio liquidated each month and repurchased with new quantities


The portfolio should be **equal-weighted**. # Can change, maybe rank?

# Signal Info

WRDS_Compustat_Fundamentals codes: naics, rdq, atq, ltq, oiadpq, req, revtq, wcapq, xoprq

Notes: current_signals is set through the processing stage.

Signals:

As our bankruptcy measure, we'll use Altman Z-Score from the COMPUSTAT Fundamental Annual dataset on WRDS:

$$\text{Altman Z-Score} = 1.2*\frac{NWC}{TA} + 1.4*\frac{RE}{TA} + 3.3*\frac{EBIT}{TA} + 0.6*\frac{\text{Equity}}{TL} + 1.0*\frac{\text{Revenue}}{TA}$$

$\frac{\text{Retained Earnings}}{\text{Total Debt}}$ gives us an outlook into how well a given company can weather hardship going forward.

EPS_Old is something that affects retail traders more, so some stocks may be a small bump and could be tradeable

Momentum (which consists of Value, Growth, Quality, and K-Score), along with Moving Average and Price Acceleration, are classic technical trading signals that have had staying power.

# TODO: Add stories for the following

## Addison
Net Income - See company revenue and performance over the years; State/Province - see location effect on "wellbeing" of stock

OA Cash flows - determine actual cash flows of activity that generate revenue, could positively correlate to market price per share and dividend

Gross margin - company might have high sales but low gross margin due to high cogs, rising GM can increase stock price

Debt to Equity - see company's shares outstanding relative to their debt, higher financing from debt may negatively impact price


## Antony
Return on Equity (ROE) – Measures how efficiently a company generates profit from shareholders' equity. Higher ROE means better profitability.


Return on Assets (ROA) – Shows how well a company turns its assets into profit. A higher ROA indicates stronger operational efficiency.


Operating Efficiency – Measures how effectively a company uses its revenue to generate profit. Higher efficiency means better cost management.


Leverage Ratio – Indicates financial risk by comparing debt to assets. A lower ratio suggests financial stability, while a higher ratio signals risk.


Interest Coverage Ratio – Shows how easily a company can pay its debt interest. A higher ratio means lower default risk.


## Ben

Total Shares repurchased-Quarter: When a company repurchases its own shares, it often signals that management believes the stock is undervalued. This can be interpreted as a sign of confidence in the company's future prospects.

In-process R&D: represents a company's investment in future product providing insights into a company's future growth potential.

Capital Expenditures: increasing capital expenditures indicate that a company is investing in growth (e.g., new facilities, equipment, or technology).

Cash Dividends: provides insights into a company's financial health, stability, and shareholder commitment. Steady dividends make a company more appealing to investors.

Deferred Revenue: represents cash received for goods or services not yet delivered. A high value demonstrates high cash flow health.

## Jackson

Debt - Looking at a companies debt at face value provides some insight into their operations.

Debt to Equities Ratio - Looking at this value provides a description of how much debt a company has versus their assets indicating how risky and growth oriented the company is.

EBITDA - Looks at a companys overall performance in earnings. This can give insight into a company's earnings before interest, taxes, depreciation, and amortization. A growing EBITDA can be a good buy signal.

Enterprise Value - Looks at the total value of a company. An increasing value can be a buy signal.

Enterprise Multiple - Compares the total value of a company to its EBITDA. Used to find a company's relative value. Companies that have a higher Enterprise Multiple means that the market has higher growth expectations while companies with lower ones have less expectations. Change in the Enterprise Multiple can indicate a buy or sell.

## Rohan
Volume - High/Low volume of shares traded each day can be indicators of strong or weak performance

Earnings Per Share - Measures a company's profitability in relation to its shareholders

P/E Ratio - Used to assess the value of a stock: stock price vs. earnings; e.g. high stock price and low earnings can be a sign of overvaluation

Relative Strength Index - Detects overbought/oversold conditions in a stock

Bollinger Bands - Stock price compared to the bands indicates volatility of the market and potential trading opportinites


## Ryan

Current Assets, Cash Flow Model, Income Taxes, Acquisitions, Depreciation & Amortization


## Ved

Profit Margin – Measures how much of a company’s revenue turns into profit after expenses.

Risk-Free Rate – The return on an investment with zero risk, often based on U.S. Treasury bonds.

Total Invested Capital – The total funds invested in a company, including debt and equity.

Net Receivables – The amount a company expects to collect from customers after deducting allowances.

Return on Invested Capital – A measure of how efficiently a company generates returns from its invested capital.

Cost of Goods Sold – The direct costs of producing goods or services sold by a company.


# <span style="color:blue">Part 1</span>: Data Processing (if needed) and setup

In [ ]:
### Imports
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display, Markdown, Latex, clear_output
import math
import scipy.stats as stats
import statsmodels.api as sm
import seaborn as sns

import concurrent.futures

# Time related things
import datetime as dt
from datetime import timedelta, date
import time # for initial_data_processor

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.inspection import PartialDependenceDisplay

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker

import gc
import os
from pathlib import Path # for initial_data_processor



# Comment in/out if needed
# import warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning)

# Suppress specific warning
# from sklearn.exceptions import DataConversionWarning
# warnings.filterwarnings("ignore", category=DataConversionWarning)

: 

In [ ]:
if not('externally_run' in locals() or 'externally_run' in globals()):
    externally_run = False

if not('missing_signal' in locals() or 'missing_signal' in globals()):
    # Inputs and Constants
    %run Background_Scripts/inputs&constants.ipynb
    # Creating our portfolios
    %run Background_Scripts/portfolio.ipynb

running data processor


/tmp/ipykernel_73449/396427500.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  price_df = pd.read_csv(price_path)


merging signal and gvkey


/home/ggrrcc/Documents/GitHub/ML-Interpretability/Background_Scripts/signal_info.py:170: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df[i].fillna(value = mean, inplace = True)
/home/ggrrcc/Documents/GitHub/ML-Interpretability/Background_Scripts/signal_info.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_df[i].fillna(value

          permno       date        PRC       RET   datadate        rdq  \
86         10001 1991-05-06   10.00000  0.000000 1991-03-31 1991-05-03   
87         10001 1991-05-07   10.00000  0.000000 1991-03-31 1991-05-03   
88         10001 1991-05-08   10.00000  0.000000 1991-03-31 1991-05-03   
89         10001 1991-05-09    9.75000 -0.025000 1991-03-31 1991-05-03   
90         10001 1991-05-10   10.00000  0.025641 1991-03-31 1991-05-03   
...          ...        ...        ...       ...        ...        ...   
55714301   93436 2020-12-24  661.77002  0.024444 2020-09-30 2020-10-21   
55714302   93436 2020-12-28  663.69000  0.002901 2020-09-30 2020-10-21   
55714303   93436 2020-12-29  665.98999  0.003465 2020-09-30 2020-10-21   
55714304   93436 2020-12-30  694.78003  0.043229 2020-09-30 2020-10-21   
55714305   93436 2020-12-31  705.66998  0.015674 2020-09-30 2020-10-21   

           naics  Future_Close  Future_Price_Change Gen_Label  ...  \
86        221210        10.000           

In [ ]:
# global portfolio_permnos_list, portfolio_keys_list
if is_full == True:
    permnos = []
    for stock in stocks:
        if stock != 'all_stocks':
            permnos.append([stock,])
else:
    permnos = [['11308'], ['12490'], ['19561'], ['52695'], ['66093'], ['66157'], ['77730']]
portfolio_permnos_list, portfolio_keys_list, portfolio_interpretability_list = portfolio_types(merged_data, permnos, signal_label_dict)

In [ ]:
portfolio_objects = []  # List to store portfolio instances

def portfolio_execution(number, extra_info=False):
    print("Portfolio", number, "is running")
    p = Portfolio(portfolio_keys_list[number], portfolio_permnos_list[number], number, missing_signal, merged_data, merged_data_train, merged_data_test, extra_info=extra_info)
    p.compute_trades()
    p.execute_trades()
    p.output_stats()
    # p.pdp_plot()
    
    portfolio_objects.append(p)  # Store the Portfolio object
    return f"Result from Portfolio {number}"

In [ ]:
p = Portfolio(portfolio_keys_list[0], portfolio_permnos_list[0], 0, missing_signal, merged_data, merged_data_train, merged_data_test, False)
p.compute_trades()
p.execute_trades()
p.output_stats()
# p.heatmap()

In [ ]:
portfolio_len = list(range(len(portfolio_keys_list)))

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = {executor.submit(portfolio_execution, num, extra_info=False): num for num in portfolio_len}
    for future in concurrent.futures.as_completed(futures):
        print(future.result())

### Print plot of alpha vs model interpretability

In [ ]:
plot_alpha_interpretability(portfolio_objects, portfolio_interpretability_list)

### Delete some vars to ensure some stuff is recalculated

In [ ]:
if not externally_run:
    del missing_signal